# Run Bi-Encoder Inference on Full Dev Set (1,942 problems)

**Tests bi-encoder alone (no verifier) with different top-k values**

**IMPORTANT: Set Runtime to GPU (T4 or A100/H100)**

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Clone repository
!git clone https://github.com/matrix-mayank/mathfish.git
%cd mathfish

In [ ]:
# 3. Install dependencies
!pip install -q torch transformers sentence-transformers datasets huggingface-hub tqdm

In [ ]:
# 4. Download FULL dev dataset (1,942 problems)
!python scripts/sample_dev.py --n 999999 --output data/processed/problems_dev_full.jsonl
!wc -l data/processed/problems_dev_full.jsonl

In [ ]:
# 5. Upload trained bi-encoder (upload biencoder_bge_large_15k.zip or biencoder_full_15k.zip)
from google.colab import files
import zipfile
import os

print("Upload your bi-encoder checkpoint:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'Extracting {filename}...')
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        if 'bge_large' in filename:
            os.makedirs('outputs/biencoder_bge_large_15k/checkpoint', exist_ok=True)
            zip_ref.extractall('outputs/biencoder_bge_large_15k/checkpoint')
            biencoder_path = 'outputs/biencoder_bge_large_15k'
            model_name = 'BGE-large'
        elif 'full' in filename:
            os.makedirs('outputs/biencoder_full_15k/checkpoint', exist_ok=True)
            zip_ref.extractall('outputs/biencoder_full_15k/checkpoint')
            biencoder_path = 'outputs/biencoder_full_15k'
            model_name = 'MiniLM-L6 (full)'
        else:
            os.makedirs('outputs/biencoder_5k_gpu/checkpoint', exist_ok=True)
            zip_ref.extractall('outputs/biencoder_5k_gpu/checkpoint')
            biencoder_path = 'outputs/biencoder_5k_gpu'
            model_name = 'MiniLM-L6 (5k)'

print(f"\n✅ Using: {model_name}")
!ls -lh {biencoder_path}/checkpoint/

In [ ]:
# 6. Run inference with multiple top-k values
import subprocess
import json

results = []

for k in [3, 5, 7, 10]:
    print(f"\n{'='*60}")
    print(f"Testing top-{k}")
    print(f"{'='*60}")
    
    output_file = f"outputs/biencoder_top{k}.jsonl"
    
    # Run inference
    subprocess.run([
        "python", "scripts/run_inference.py",
        "--checkpoint_dir", f"{biencoder_path}/checkpoint",
        "--problems_file", "data/processed/problems_dev_full.jsonl",
        "--from_hf",
        "--output_file", output_file,
        "--top_k", str(k)
    ], check=True)
    
    # Evaluate
    result = subprocess.run([
        "python", "scripts/evaluate_alignment.py",
        "--predictions_file", output_file,
        "--gold_file", "data/processed/problems_dev_full.jsonl"
    ], capture_output=True, text=True)
    
    metrics = json.loads(result.stdout)
    metrics['top_k'] = k
    results.append(metrics)
    
    print(f"\n📊 Results:")
    print(f"  Exact Match: {metrics['exact_match']:.4f}")
    print(f"  Micro F1: {metrics['micro_f1']:.4f}")
    print(f"  Macro F1: {metrics['macro_f1']:.4f}")
    print(f"  Recall@5: {metrics['recall@5']:.4f}")
    print(f"  Recall@20: {metrics['recall@20']:.4f}")

# Summary
print(f"\n{'='*80}")
print("📋 FINAL SUMMARY")
print(f"{'='*80}")
print(f"\n{'Top-K':<8} {'Exact':<10} {'Micro F1':<12} {'Macro F1':<12} {'Recall@5':<12} {'Recall@20':<12}")
print("-" * 80)
for r in results:
    print(f"{r['top_k']:<8} {r['exact_match']:<10.4f} {r['micro_f1']:<12.4f} {r['macro_f1']:<12.4f} {r['recall@5']:<12.4f} {r['recall@20']:<12.4f}")

# Find best
best_exact = max(results, key=lambda x: x['exact_match'])
best_f1 = max(results, key=lambda x: x['micro_f1'])
print(f"\n🎯 Best Exact Match: top-{best_exact['top_k']} ({best_exact['exact_match']:.4f})")
print(f"🎯 Best Micro F1: top-{best_f1['top_k']} ({best_f1['micro_f1']:.4f})")

In [ ]:
# 7. Download results
import shutil

shutil.make_archive('biencoder_inference_results', 'zip', 'outputs')
files.download('biencoder_inference_results.zip')

print(f"\n✅ Results for {model_name} on 1,942 dev problems")